# Voortoets ANB (Vlaamse Overheid)

This notebook uses hantush_compromise.py for most of its code.

The idea is to compare a model with actual ditches with one in which these
ditches are replaced by a semi-permeable layer on top of the aquifer.

Three models are used:

1) 1D model with diches 500 m apart simulated by GHB and its equivalent semi-permeable top layer.
2) 2D (quasi 3D) model with the same ditches but a well in the center.
3) 2D (quasi 3D) model with actual surface water somewhere in Flanders.

The essencd of the matter is the question as to what extent Hantush (or De Glee) can be used
to simulate drawdown and influence radius instead of a model with an actual network of
water courses. That's why the file is name "hantush-compromise"

The report was sent to AGT on 2026-03-30

The report:
"/Users/Theo/Entiteiten/Hygea/2022-AGT-Voortoets/Coding/

@TO 2026-03-30

In [1]:
import os
import sys
from itertools import cycle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import k0 as K0
from tools.fdm.src.fdm3 import fdm3
from tools.fdm.src.mfgrid import Grid

import hantush_compromise as hc
import vtl_surf_water as sw

parts = Path(os.getcwd()).parts
assert '2022-AGT-Voortoets' in parts, "2022-AGT must be in path for correct saving: {home}"
images = os.path.join(*parts[:parts.index("2022-AGT-Voortoets") + 1], "Coding", "images")


print(sys.executable)
print(os.getcwd())

loading mfpath.py
/Users/Theo/Entiteiten/Hygea/2022-AGT-Voortoets/Coding/.venv/bin/python
/Users/Theo/Entiteiten/Hygea/2022-AGT-Voortoets/Coding/.venv/bin/python
/Users/Theo/Entiteiten/Hygea/2022-AGT-Voortoets/Coding/src


In [2]:
well_coordinates = hc.well_coordinates

L = 500. # --- Distance between ditches
D = 25.  # --- Aquifer thickness
k = 10.  # --- Hydraulic conductivity of aquifer

# --- Ditch resistance, x of the well, extraction of well
w = 0.5  # --- Ditch resistance [d/m]
xw = 0.  # --- Location of well [m]

# --- 4 values of the ditch radial resistance.
ws=[0.25, 0.5, 1, 2] # --- list of ditch resistances [d/m]

# --- Radius of the ditch cross section
rDitch = 1

## The 1D test model comparing parallel ditches to Mazure's solution

In [ ]:
Qw = -1.
hc.two_simlataneous_3Dmdoels(L=L, D=D, k=k, ws=ws, xw=xw, Qw=Qw, rDitch=1)

## The 2D (quasi-3D) model comparing parallel ditches to De Glee's solution

In [ ]:
Qw = -2400.
hc.two_simultaneous_3Dmodels(L=L, D=D, k=k, ws=ws, xw=xw, Qw=Qw, rDitch=rDitch)

## The realworld solution using Fdm3 with actual surface water at locations in Flanders

In [ ]:
for wel in well_coordinates:
    xy, Q, b = wel[0], wel[1], wel[2]
    Lmdl, D, k, w, dxy = 10000, 20, 10, 2.5, 50
    print(f"Running well Q={Q} at {xy[0]}{xy[1]}, b={b} m")
    hc.fdm_real_surfwater(xy=xy, Q=Q, b=b, Lmdl=Lmdl, D=D, k=k, dxy=dxy, z=[0, -2, -2 - D], w=w)
print("Done")


## Generate a map showing the distance to surface water, plus well locations

The distance map is generated for the area around a location in Flanders with the
surface water taken form the open map.

well_coordinates are chosen and shown on the map together with the max. distance to surface
at teh well locations. These well locations are used to generate finite difference models
to compare the generated fd model with actual surface water to the analytical drawdown
according to De Glee (i.e. with surface water replaced by an equivalent semi-permeable
layer on top of the aquifer).

# Generate a model at given location..

In [ ]:
water_gdf = sw.clip_water_to_gr(gr)
sw.distance_raster(water_gdf, pixelsize=10)

dist_to_water = sw.distance_grid_with_grates(xy=(194000, 195000), L=15000, dxy=50)
fig = plt.gcf()
ax = fig.axes[0]
for obs in sw.well_coordinates:
    path = sw.climb_to_ridge(*obs, dist=dist_to_water)[[0, -1]]
    ax.plot(*path.T, 'r-')
    ax.plot(*path[0], 'o', ms=7, mec='r', mfc='white', alpha=1)
    ax.plot(*path[1], 'o', ms=7, mec='r', mfc='gold',  alpha=1)
fig.savefig(os.path.join(images, "afstand_tot_oppwater.png"))

In [ ]:
for wel in sw.well_coordinates:
    xy, Q, b = wel[0], wel[1], wel[2]
    L, D, k, w, dxy = 10000, 20, 10, 2, 50
    print(f"Running well Q={Q} at {xy[0]}{xy[1]}, b={b} m")
    sw.fdm_real_surfwater(xy=xy, Q=Q, b=b, L=L, D=D, k=k, dxy=dxy, z=[0, -2, -2 - D], w=w)
print("Done")